# EDGAR Financial Sentiment — Part 4: QLoRA Fine-Tuning of Mistral-7B

Adapted from **Assignment 3** (QLoRA), retargeted to finance.

**The idea:** Parts 2–3 fine-tuned *small* models in full. Here we fine-tune a **7-billion-parameter** model — `mistralai/Mistral-7B-Instruct-v0.3` — on the same Financial PhraseBank task, on a single T4. That's only possible with **QLoRA**: load the base model in **4-bit** (frozen), and train a tiny set of **LoRA adapter** matrices on top (<1% of the parameters). Same benchmark as before, so the result drops straight into the eventual bake-off.

**What you'll practice:** you chose the *hybrid* build — I provide the QLoRA model setup (the new concept), and **you write the `Dataset` and the hand-written train / test loops**, exactly like Parts 2–3. The one new wrinkle in the loop: the model here is a Hugging Face classifier, so you call it with keyword args and read `out.logits` (no custom `nn.Module` wrapper this time).

> **Run in Google Colab with a T4 GPU.** Expect ~8 min to download the weights and ~20–40 min to train.

### QLoRA in one paragraph

**LoRA** freezes the original weight matrices `W` and learns a small low-rank update `BA` (rank `r`), so each adapted layer computes `W x + B(A x)`. Only `A` and `B` train — a few million params instead of seven billion. **QLoRA** adds the `Q`: the frozen `W` is stored in **4-bit** (NF4) to slash memory, while the LoRA matrices stay in higher precision. The result fits a 7B model + its gradients on a 16 GB T4. We attach the adapters to the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`).

## 0. Setup

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets

In [ ]:
import random, io, zipfile, requests
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset as HFDataset, ClassLabel   # aliased so it doesn't shadow torch's Dataset

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'Switch the Colab runtime to a T4 GPU before running.'

### Hugging Face login (Mistral is gated)
`mistralai/Mistral-7B-Instruct-v0.3` is a **gated** model. Before this notebook can download it you must:
1. Accept the license at <https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3>
2. Make a token at <https://huggingface.co/settings/tokens>
3. Run the cell below and paste the token (or store it once via Colab *Secrets* &rarr; `HF_TOKEN`).

In [ ]:
from huggingface_hub import login
login()   # paste your HF token when prompted

## 1. Config & seeds

In [ ]:
torch.manual_seed(10); random.seed(10); np.random.seed(10)

MODEL_ID        = 'mistralai/Mistral-7B-Instruct-v0.3'
NUM_CLASSES     = 3
max_len         = 64
batch_size      = 4       # small: 4-bit base + gradient checkpointing keep this comfortable on a T4
eval_batch_size = 8
lr              = 2e-4    # typical LoRA learning rate (higher than full fine-tuning)
TRAIN_STEPS     = 500     # ~1 epoch over PhraseBank at batch_size 4; raise for more

# LoRA hyperparameters
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05

## 2. Load the Financial PhraseBank benchmark

Same data + stratified 80/20 split as Parts 2–3, from the raw zip.

In [ ]:
_URL = 'https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip'
_z = zipfile.ZipFile(io.BytesIO(requests.get(_URL, timeout=120).content))
_member = next(n for n in _z.namelist() if n.endswith('Sentences_AllAgree.txt'))
_raw = _z.read(_member).decode('latin-1')

_label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
_rows = []
for _line in _raw.splitlines():
    _line = _line.strip()
    if not _line:
        continue
    _sent, _lab = _line.rsplit('@', 1)
    _rows.append({'sentence': _sent, 'label': _label_map[_lab.strip()]})
_df = pd.DataFrame(_rows)

pb_full = HFDataset.from_pandas(_df).cast_column('label', ClassLabel(names=['negative', 'neutral', 'positive']))
pb = pb_full.train_test_split(test_size=0.2, seed=10, stratify_by_column='label')
pb_train, pb_test = pb['train'], pb['test']
label_names = pb_train.features['label'].names
print('Label index -> name:', dict(enumerate(label_names)))
print('Train size:', len(pb_train), '| Test size:', len(pb_test))
print('Example row:', pb_train[0])

## 3. Load Mistral-7B in 4-bit and attach LoRA adapters  *(provided — the new concept)*

This is the QLoRA machinery. Read it closely; the tunable knobs are in the config cell above.

**Tokenizer note:** Mistral has no pad token, so we reuse its end-of-sequence token for padding. And we pad on the **right** here — the opposite of Part 2. Why? The sequence-classification head finds the *last non-pad token* assuming right-padding, then classifies from that position. (In Part 2 we read the last position ourselves, so we padded left; here the model does it, so we pad right.)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token   # Mistral ships without a pad token
tokenizer.padding_side = 'right'            # SeqClassification locates the last real token assuming right-pad

In [ ]:
# --- 4-bit quantization (the 'Q' in QLoRA) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',             # normal-float-4: a good default for LLM weights
    bnb_4bit_use_double_quant=True,        # quantize the quantization constants too
    bnb_4bit_compute_dtype=torch.float16,  # T4 has no bf16 -> use fp16 for the matmuls
)

# Load Mistral-7B as a 3-way sequence classifier, weights quantized to 4-bit.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=NUM_CLASSES,
    quantization_config=bnb_config, device_map={'': 0},
)
model.config.pad_token_id = tokenizer.pad_token_id   # let the head find the last real token
model.config.use_cache = False                       # required with gradient checkpointing

# Freeze the 4-bit base; enable gradient checkpointing + input grads for memory-efficient training.
model = prepare_model_for_kbit_training(model)

# --- LoRA adapters (the 'LoRA' in QLoRA): small trainable matrices on the attention projections ---
lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias='none',
    task_type=TaskType.SEQ_CLS,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # expect well under 1% of params trainable

loss_fn = nn.CrossEntropyLoss()

## 4. Build the `Dataset`  &larr; **your code**

Same shape as Part 3 (good reinforcement): tokenize each sentence with right-padding and keep the class index as the label. Tensors stay on **CPU**; the loops move each batch to the GPU. Store per row:
- `input_ids` `(max_len,)`, `attention_mask` `(max_len,)`, `label` (scalar `int64`).

In [ ]:
class ClassificationData(Dataset):
    """Tokenize (sentence, label) rows for Mistral sequence classification.

    Right-padding (the classification head locates the last non-pad token that way).
    Per row store 'input_ids', 'attention_mask' (max_len,) and 'label' (scalar int64),
    all on CPU.
    """
    def __init__(self, hf_split, tokenizer, max_len):
        self.data = []
        for ex in hf_split:
            ### YOUR CODE HERE
            # 1. tokenize ex['sentence'] with: max_length=max_len, truncation=True,
            #    padding='max_length', return_tensors='pt'
            # 2. append a CPU dict to self.data with 'input_ids', 'attention_mask'
            #    (each .squeeze(0)) and 'label' = torch.tensor(ex['label'], dtype=torch.int64)
            ...
            ### END YOUR CODE

    def __len__(self):
        ### YOUR CODE HERE
        ...
        ### END YOUR CODE

    def __getitem__(self, index):
        ### YOUR CODE HERE
        ...
        ### END YOUR CODE

### Sanity-check your `Dataset`
The first token should be Mistral's beginning-of-sequence token `<s>`.

In [ ]:
_probe = ClassificationData(pb_train, tokenizer, max_len)
print('Examples built:', len(_probe))
_ex = _probe[0]
print('input_ids shape:', tuple(_ex['input_ids'].shape), '| real tokens:', int(_ex['attention_mask'].sum()))
print('first token:', repr(tokenizer.decode([int(_ex['input_ids'][0])])), '(Mistral BOS)')
print('label:', int(_ex['label']), '->', label_names[int(_ex['label'])])
print('Decoded (real tokens):', repr(tokenizer.decode(_ex['input_ids'][_ex['attention_mask'].bool()])))

## 5. Train and test loops  &larr; **your code**

Same structure as Part 3, with **one difference**: there's no custom `nn.Module` wrapper here. `model` is the Hugging Face classifier, so you call it with keyword args and read the logits off the output object:

```python
out = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
logits = out.logits          # shape (batch, num_classes)
```

Then compute the loss against `batch['label']` yourself with `loss_fn`, exactly as before.

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer, reporting_interval=50, steps=None):
    """One pass of QLoRA fine-tuning.

    - model.train(); track running loss + batch count
    - for each batch (stop after `steps`):
        - move the batch dict to device
        - out = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
        - logits = out.logits  -> (batch, num_classes); targets = batch['label']
        - zero_grad, CrossEntropyLoss, backward, optimizer.step()
        - accumulate loss; print running avg every `reporting_interval` batches
    """
    ### YOUR CODE HERE
    # follow the steps in the docstring above
    ...
    ### END YOUR CODE

In [ ]:
def test_loop(dataloader, model, loss_fn, steps=None):
    """Evaluate accuracy: argmax(logits) == label.

    - model.eval(); torch.no_grad()
    - per batch (stop after `steps`): move to device; out = model(input_ids=..., attention_mask=...);
      logits = out.logits; add loss; count correct (argmax == label) and total
    - print accuracy + avg loss; return accuracy in percent
    """
    ### YOUR CODE HERE
    ...
    ### END YOUR CODE

## 6. Run QLoRA fine-tuning

The optimizer only sees the **trainable** (LoRA) parameters — the 4-bit base stays frozen. One pass of `TRAIN_STEPS` batches, then evaluate on the held-out test set.

In [ ]:
train_loader = DataLoader(ClassificationData(pb_train, tokenizer, max_len),
                          batch_size=batch_size, shuffle=True, drop_last=True)
test_loader  = DataLoader(ClassificationData(pb_test, tokenizer, max_len),
                          batch_size=eval_batch_size, shuffle=False)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)

print('QLoRA fine-tuning Mistral-7B...')
train_loop(train_loader, model, loss_fn, optimizer, steps=TRAIN_STEPS)
print('Evaluating...')
acc = test_loop(test_loader, model, loss_fn)
print(f'\n================ RESULT ================')
print(f'Mistral-7B (QLoRA) test accuracy: {acc:.1f}%')

## 7. What to look for

- **Did a 7B model beat the small ones?** Compare against your Part-2 GPT-2 (~82–86%) and Part-3 BERT/FinBERT numbers. A QLoRA-tuned Mistral usually lands at or above FinBERT — but not always by much on a task this clean, which is itself a useful observation about diminishing returns.
- **`print_trainable_parameters` is the headline of QLoRA:** typically a fraction of a percent of the 7B params are trainable, yet you get competitive accuracy. That ratio is the whole point.
- **Padding side flipped back to right** (vs. Part 2's left) because the HF classification head locates the last real token assuming right-padding. Same underlying idea (classify from the last token), different owner of the bookkeeping.
- **Things to try:** raise `TRAIN_STEPS`; change LoRA `r` / `target_modules` (add the MLP projections `gate_proj`/`up_proj`/`down_proj`); or swap to the non-instruct base `mistralai/Mistral-7B-v0.3`.

**Next (Track B):** Part 5 — **LLM-as-classifier**: prompt this same Mistral *zero-shot* and *few-shot* with no training at all, as a no-fine-tuning baseline against everything above.